# Pertemuan 04 — K-Means dan Probabilitas

Pada modul ini kita akan mempelajari dua hal:

1. **K-Means Clustering** — sebuah teknik *unsupervised learning* untuk mengelompokkan data yang **tidak berlabel**.
2. **Perhitungan probabilitas dasar** — khususnya penerapan **Teorema Bayes** pada kasus pendeteksi kata bangun (*wake word*).

> **Beda dengan pertemuan sebelumnya.** Pada Pertemuan 03 kita mengerjakan *supervised learning*: setiap contoh data punya label, dan model belajar memetakan fitur ke label. Di sini datanya **tidak punya label sama sekali**. Tugas algoritma adalah menemukan sendiri kelompok-kelompok alami di dalam data.

### Apa yang akan kita kerjakan

| Bagian | Isi |
|---|---|
| 1 | K-Means dengan scikit-learn pada data bersepeda |
| 2 | Membangun sendiri algoritma K-Means dari nol (algoritma Lloyd) |
| 3 | K-Means pada data piksel untuk kompresi warna gambar |
| 4 | Memilih jumlah klaster dengan metode siku (*elbow method*) |
| 5 | Teorema Bayes pada pendeteksi wake word |

> **Cara memakai modul ini.** Jalankan sel kode berurutan dari atas ke bawah. Banyak sel bergantung pada hasil sel sebelumnya, sehingga melompati satu sel akan menyebabkan error.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly import figure_factory as ff
colors = px.colors.qualitative.Plotly
px.defaults.width = 800
from ipywidgets import HBox
import numpy as np
pd.set_option('plotting.backend', 'plotly')

In [ ]:
# Membuat folder 'images' bila belum ada
import os
if not os.path.exists("images"):
    os.makedirs("images")

# # Hapus tanda pagar bila ingin mengekspor ke HTML
# import plotly.io as pio
# pio.renderers.default = "notebook_connected"

---
## 1. Dataset Bersepeda

Kita akan menerapkan k-means clustering pada data bersepeda untuk menelusuri sebaran **panjang lintasan** (*Length*) dan **kecepatan** (*Speed*) dari perjalanan bersepeda.

Setiap baris data mewakili satu segmen perjalanan. Kita tahu data ini berasal dari **4 sepeda yang berbeda**, tetapi informasi sepeda mana yang dipakai **tidak tercatat** pada datanya. Pertanyaannya: dapatkah algoritma menemukan sendiri keempat kelompok itu hanya dari pola kecepatan dan panjang lintasannya?


In [ ]:
# bikes = pd.read_csv("speed_length_data.csv")
bikes = pd.read_csv("speed_length_data.csv")
bikes.head()

In [ ]:
bikes.plot.scatter(x='Speed', y='Length', title='Kecepatan vs Panjang Segmen Perjalanan',
                   height=800)

Sekilas pada grafik pencar di atas sudah terlihat adanya beberapa gerombolan titik. Tugas kita berikutnya adalah membuat algoritma yang menemukan gerombolan itu secara otomatis.

> **Perhatikan.** Kolom `Speed` dan `Length` memiliki satuan dan rentang yang berbeda. Karena k-means bekerja berdasarkan **jarak**, perbedaan skala dapat memengaruhi hasil. Hal ini menjadi bahan latihan mandiri di akhir modul.

### 1.1 K-Means dengan Scikit-Learn

Karena kita tahu datanya berasal dari 4 sepeda, kita coba k-means dengan **4 klaster**.

Perhatikan perbedaan penting dengan model *supervised* pada Pertemuan 03: di sini `fit()` hanya menerima **X saja**, tanpa **y**. Tidak ada label yang diberikan kepada algoritma.

Cara kerja k-means secara ringkas: algoritma mencari `k` titik pusat (*centroid*) sedemikian rupa sehingga jumlah kuadrat jarak setiap titik data ke pusat terdekatnya sekecil mungkin.


In [ ]:
from sklearn.cluster import KMeans
# Membuat model KMeans dengan 4 klaster
kmeans = KMeans(n_clusters=4, random_state=42)
# Melatih model pada data (perhatikan: tanpa label y!)
kmeans.fit(bikes[['Speed', 'Length']])
# Mengambil nomor klaster untuk tiap titik data
bikes['scikit k-means'] = kmeans.predict(bikes[['Speed', 'Length']]).astype(str)

Sekarang mari kita visualkan hasil pengelompokannya. Titik hitam menandai posisi pusat (*centroid*) setiap klaster.

In [ ]:
fig = px.scatter(
    bikes, x='Speed', y='Length', color='scikit k-means',
    title='Hasil K-Means pada Segmen Perjalanan',
    height=800)
fig.add_scatter(
    x=kmeans.cluster_centers_[:,0],
    y=kmeans.cluster_centers_[:,1],
    mode='markers',
    marker=dict(color='black', size=10),
    name='Pusat klaster'
)
#fig.write_image("images/bike_kmeans.pdf", scale=2, height=800, width=700)


> **Amati hasilnya.** Apakah keempat klaster yang ditemukan algoritma terlihat masuk akal? Perhatikan bahwa k-means menghasilkan batas antar-klaster yang berupa **garis lurus** — ini konsekuensi dari pemakaian jarak Euclidean.

---
*Kembali ke slide.*

---


---
## 2. Membangun Sendiri Algoritma K-Means

Cara terbaik memahami sebuah algoritma adalah menuliskannya sendiri. **Algoritma Lloyd** untuk k-means terdiri atas tiga bagian utama:

| Tahap | Yang dikerjakan |
|---|---|
| **Inisialisasi** | Menentukan posisi awal `k` pusat klaster |
| **Penugasan** (*assignment*) | Menetapkan setiap titik data ke pusat terdekat |
| **Pembaruan** (*update*) | Menggeser setiap pusat ke rata-rata titik anggotanya |

Tahap penugasan dan pembaruan diulang bergantian sampai penugasannya tidak berubah lagi. Kita akan menulis satu fungsi untuk setiap tahap.


### 2.1 Inisialisasi

Kita memakai **metode Forgy**, yaitu memilih `k` titik data secara acak untuk dijadikan pusat awal.

> **Mengapa pemilihan awal penting?** K-means tidak menjamin solusi terbaik secara global. Pusat awal yang berbeda dapat menghasilkan pengelompokan akhir yang berbeda pula. Itulah sebabnya scikit-learn secara bawaan menjalankan algoritmanya beberapa kali dengan inisialisasi berbeda, lalu memilih hasil terbaik.


In [ ]:
def initialize_centers(x, k):
    """Memilih k titik berbeda secara acak dari x sebagai pusat awal."""
    ind = np.random.choice(np.arange(x.shape[0]), k, replace=False)
    return x[ind]

In [ ]:
# Ambil kolom Speed dan Length sebagai array NumPy, lalu pilih pusat awal secara acak
k = 4
x = bikes[['Speed', 'Length']].to_numpy()
centers = initialize_centers(x, k)
centers

### 2.2 Penugasan (Assignment)

Pada tahap ini, setiap titik data ditetapkan ke pusat klaster yang **paling dekat**.

Perhitungannya memakai jarak Euclidean. Baris `x[:, np.newaxis] - centers` adalah teknik *broadcasting* NumPy: ia menghitung selisih setiap titik terhadap setiap pusat sekaligus, tanpa perlu perulangan bersarang. Hasilnya berupa matriks jarak berukuran (jumlah titik × jumlah pusat), lalu `argmin` mengambil indeks pusat terdekat untuk tiap titik.


In [ ]:
def compute_assignments(x, centers):
    """Menetapkan setiap titik pada x ke pusat klaster terdekat."""
    distances = np.linalg.norm(x[:, np.newaxis] - centers, axis=2)
    return np.argmin(distances, axis=1)

In [ ]:
# Setiap angka menunjukkan nomor pusat klaster terdekat bagi tiap titik data
assignments = compute_assignments(x, centers)
assignments

### 2.3 Pembaruan Pusat (Update)

Pada tahap ini setiap pusat klaster digeser ke **rata-rata** dari semua titik yang ditugaskan kepadanya.

Inilah asal nama *k-means*: pusat klaster adalah nilai rata-rata (*mean*) anggotanya.


In [ ]:
def update_centers(x, assignments, k):
    """Memperbarui pusat klaster berdasarkan penugasan saat ini."""
    return np.array([x[assignments == i].mean(axis=0) for i in range(k)])

In [ ]:
# Pusat digeser ke rata-rata anggotanya. Bandingkan dengan pusat sebelumnya di atas.
centers = update_centers(x, assignments, k)
centers

Bandingkan keluaran ini dengan pusat awal pada sel sebelumnya — posisinya sudah bergeser.

Satu putaran penugasan dan pembaruan seperti ini disebut **satu iterasi**. Berikutnya kita ulang iterasi tersebut sampai posisinya stabil.

### 2.4 Algoritma Lloyd Lengkap

Sekarang kita rangkai ketiga fungsi tadi dalam sebuah perulangan yang berhenti ketika **penugasan tidak berubah lagi** — artinya algoritma sudah konvergen.

Variabel `soln_path` menyimpan posisi pusat pada setiap iterasi, sehingga nantinya kita bisa membuat animasi prosesnya.

> **Mengapa pasti berhenti?** Setiap langkah penugasan dan pembaruan selalu menurunkan (atau paling tidak tidak menaikkan) nilai fungsi tujuan k-means. Karena jumlah kemungkinan penugasan terbatas, algoritma pasti mencapai kondisi berhenti.


In [ ]:
def k_means_clustering(x, k, max_iters=100):
    centers = initialize_centers(x, k)
    assignments_old = -np.ones(x.shape[0])
    soln_path = [centers]
    for _ in range(max_iters):
        assignments = compute_assignments(x, centers)
        centers = update_centers(x, assignments, k)
        soln_path.append(centers)
        if np.array_equal(assignments, assignments_old):
            break
        assignments_old = assignments
    return centers, assignments, soln_path

In [ ]:
# np.random.seed membuat pemilihan pusat awal selalu sama, sehingga hasilnya dapat diulang
np.random.seed(43)
centers, assignments, soln_path = k_means_clustering(x, k)
len(soln_path)

### 2.5 Animasi Proses K-Means

Kode berikut membuat animasi proses pengelompokan pada setiap iterasi. Anda **tidak perlu memahami detail kodenya** — fokuslah pada animasinya.

> **Cara menikmatinya.** Tekan tombol *Play* di bawah grafik. Perhatikan bagaimana tanda silang hitam (pusat klaster) bergerak dari posisi acak awal menuju posisi stabil, sementara warna titik-titik berubah mengikuti pusat terdekatnya.


In [ ]:
### Membuat animasi proses algoritma k-means.
### Anda tidak perlu memahami kode di bawah ini.
### Kode ini semata-mata untuk membuat animasinya.

## Menyusun satu tabel besar berisi seluruh data dan pusat, ditandai nomor iterasinya.
pts = []
for i, centers in enumerate(soln_path):
    df = bikes[['Speed', 'Length']].copy()
    df['Class'] = compute_assignments(x, centers).astype(str)
    df2 = pd.DataFrame(centers, columns=['Speed', 'Length'])
    df2['Class'] = 'Center'
    df_combined = pd.concat([df, df2], ignore_index=True)
    # Indeks tiap titik diperlukan agar animasi dapat melacak titik yang sama
    # antar-bingkai (frame)
    df_combined.reset_index(inplace=True)
    # Nomor iterasi menentukan bingkai keberapa pada animasi
    df_combined['Iteration'] = i
    pts.append(df_combined)
# Menumpuk seluruh data menjadi satu tabel besar.
frames = pd.concat(pts, ignore_index=True)

## Membuat animasinya
fig = px.scatter(frames, x='Speed', y='Length', color='Class',
                 animation_group='index',
                 animation_frame='Iteration', title='Proses Algoritma K-Means',
                 width=700, height=800)
## Menyamakan skala kedua sumbu agar jaraknya tidak menyesatkan.
fig.update_layout(
    xaxis=dict(scaleanchor="y", scaleratio=1),
    yaxis=dict(scaleanchor="x", scaleratio=1)
)
# fig.write_image("images/bike_kmeans_animation_0.pdf", height=800, width=700)

## Mempertegas tampilan pusat klaster agar mudah terlihat
fig.update_traces(marker=dict(size=12, symbol='x', color='black'),
                  selector=dict(legendgroup='Center') )
for i, f in enumerate(fig.frames):
    for trace in f.data:
        if trace.name == 'Center':
            trace.update(marker=dict(size=12, symbol='x', color='black'))
    # go.Figure(f.data, f.layout).write_image(
    #     f"images/bike_kmeans_animation_{i+1}.pdf", height=800, width=700)
# fig.write_html("images/bike_kmeans_animation.html",include_plotlyjs='cdn', full_html=True)
fig


> **Perhatikan.** Perubahan terbesar terjadi pada beberapa iterasi pertama, setelah itu pergerakannya sangat kecil. Pola ini khas pada algoritma k-means.

---
*Kembali ke slide.*

---


---
## 3. K-Means pada Data Piksel

Sekarang kita terapkan k-means pada persoalan yang sama sekali berbeda: **kompresi warna gambar**.

Gagasannya sederhana. Sebuah foto bisa mengandung ratusan ribu warna berbeda. Bila kita mengelompokkan semua warna itu menjadi hanya 8 kelompok, lalu mengganti setiap piksel dengan warna pusat kelompoknya, kita memperoleh gambar yang mirip aslinya tetapi hanya memakai 8 warna.


Mari kita muat sebuah foto perjalanan bersepeda.

In [ ]:
from PIL import Image
import requests
from io import BytesIO
url = "https://eecs189.org/fa25/resources/assets/lectures/lec04/bike2.jpeg"
response = requests.get(url)
img = np.array(Image.open(BytesIO(response.content)))
# img = np.array(Image.open("bike2.jpeg"))
print(img.shape)
px.imshow(img)

Sebuah gambar dapat kita pandang sebagai **kumpulan piksel**, dan setiap piksel diwakili oleh sebuah nilai warna.

Pada gambar RGB, setiap piksel diwakili tiga angka yang bersesuaian dengan kanal warna **merah (R), hijau (G), dan biru (B)**. Artinya setiap piksel adalah sebuah **vektor tiga dimensi**.

Karena berdimensi tiga, vektor-vektor itu dapat kita gambar dalam ruang 3D. Sel berikut memplot 100.000 piksel sebagai titik dalam ruang warna, masing-masing diwarnai sesuai warna aslinya.

> **Amati sebarannya.** Titik-titik tidak tersebar merata memenuhi kubus, melainkan menggerombol di beberapa daerah. Gerombolan itulah yang akan ditemukan oleh k-means.


In [ ]:
image_df = pd.DataFrame(img.reshape(-1,3), columns=['R', 'G', 'B'])
image_df['color'] = ("rgb(" +
                     image_df['R'].astype(str) + "," +
                     image_df['G'].astype(str) + "," +
                     image_df['B'].astype(str) + ")")
fig = go.Figure()
small_image_df = image_df.sample(100000, random_state=42)
fig.add_scatter3d(x=small_image_df['R'], y=small_image_df['G'], z=small_image_df['B'],
                   mode='markers', marker=dict(color=small_image_df['color'], opacity=0.5, size=2))
fig.update_layout(scene=dict(xaxis_title='R', yaxis_title='G', zaxis_title='B'),
                  width=800, height=800,)
# fig.write_html("images/bike_color_space.html",include_plotlyjs='cdn', full_html=True)
fig

### 3.1 Menerapkan K-Means pada Warna

Kita kelompokkan seluruh piksel menjadi **8 klaster warna**. Setiap piksel kemudian diganti dengan warna pusat klasternya.

Hasilnya adalah bentuk sederhana dari **kompresi gambar**: alih-alih menyimpan tiga angka penuh untuk setiap piksel, cukup disimpan nomor klasternya (0 sampai 7) beserta satu tabel berisi 8 warna.


In [ ]:
# Setiap piksel dikelompokkan berdasarkan warnanya (R, G, B)
from sklearn.cluster import KMeans
# Menerapkan k-means pada kolom R, G, dan B
n_clusters = 8
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
image_df['cluster'] = kmeans.fit_predict(image_df[['R', 'G', 'B']])
image_df['cluster'].value_counts()

In [ ]:
from plotly.subplots import make_subplots
img_kmeans = (
    kmeans.cluster_centers_[image_df['cluster'].values]
    .reshape(img.shape)
)
# Membuat dua panel gambar yang saling terhubung
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Gambar Asli", "Hasil Kompresi K-Means"),
    specs=[[{"type": "xy"}, {"type": "xy"}]]
)
fig.add_trace(px.imshow(img).data[0], row=1, col=1)
fig.add_trace(px.imshow(img_kmeans).data[0], row=1, col=2)

# 1) Menyamakan rentang sumbu kedua panel
#    (batas setengah piksel agar sesuai cara Plotly menggambar citra)
H, W = img.shape[:2]
xrange = [-0.5, W - 0.5]
yrange = [H - 0.5, -0.5]  # titik asal di bagian atas
for c in (1, 2):
    fig.update_xaxes(range=xrange, row=1, col=c)
    fig.update_yaxes(range=yrange, row=1, col=c)

# 2) Mengunci piksel agar tetap persegi dan tidak melar
fig.update_yaxes(scaleanchor="x",  scaleratio=1, constrain="domain", row=1, col=1)
fig.update_yaxes(scaleanchor="x2", scaleratio=1, constrain="domain", row=1, col=2)

# Menautkan geser dan perbesar antara kedua gambar
fig.update_xaxes(matches="x")
fig.update_yaxes(matches="y")

# Perapian tampilan
# fig.write_html("images/bike_kmeans_compression.html",include_plotlyjs='cdn', full_html=True)
fig.update_layout(width=900, height=600, margin=dict(t=50, b=30, l=20, r=20))


**Bandingkan kedua gambar di atas.** Gambar kanan hanya memakai 8 warna, tetapi bentuk dan isinya masih dikenali dengan jelas.

Berapa besar penghematannya? Gambar asli menyimpan 3 angka (0–255) untuk setiap piksel. Hasil kompresi cukup menyimpan satu nomor klaster (0–7, hanya perlu 3 bit) untuk setiap piksel, ditambah satu tabel kecil berisi 8 warna. Inilah gagasan dasar di balik format gambar berpalet warna seperti GIF.

---
## 4. Memilih Jumlah Klaster

Pertanyaan yang selalu muncul pada k-means: **berapa nilai `k` yang sebaiknya dipakai?**

Berbeda dengan klasifikasi, di sini tidak ada label yang dapat dijadikan acuan kebenaran. Yang bisa kita ukur adalah **fungsi tujuan k-means**, yang di scikit-learn disebut **inertia**: jumlah kuadrat jarak setiap titik ke pusat klasternya. Semakin kecil nilainya, semakin rapat pengelompokannya.

> **Kenapa tidak sekadar memilih `k` sebesar mungkin?** Karena inertia **selalu turun** ketika `k` bertambah. Pada kasus ekstrem, bila `k` sama dengan jumlah titik data, setiap titik menjadi pusatnya sendiri dan inertia menjadi nol — pengelompokan yang sempurna sekaligus tidak berguna sama sekali.

Cara baku memilih `k` adalah **metode siku** (*elbow method*): kita plot nilai fungsi tujuan terhadap `k`, lalu mencari titik "siku" tempat laju perbaikannya mulai melambat. Setelah titik itu, menambah klaster memberi keuntungan yang semakin kecil dibanding tambahan kerumitannya.


In [ ]:
scores = pd.DataFrame(columns=['k'])
scores['k'] = [2, 4, 8, 16, 32, 64, 128, 256]
scores.set_index('k', inplace=True)

# Menerapkan k-means pada kolom R, G, dan B
from sklearn.cluster import KMeans
for k in scores.index:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(image_df[['R', 'G', 'B']])
    # scores.loc[k, 'score'] = kmeans.score(image_df[['R', 'G', 'B']])
    scores.loc[k, 'score'] = kmeans.inertia_   # inertia: makin kecil makin rapat klasternya

In [ ]:
fig = px.line(
    scores,
    title="Fungsi Tujuan K-Means terhadap Jumlah Klaster",
    markers=True,
    labels={"index": "Jumlah Klaster (k)",
            "value": "Fungsi Tujuan (Inertia)"},
    width=700, height=400
)
fig.update_layout(xaxis_type="log", xaxis_exponentformat='power', showlegend=False)
fig.update_layout(margin=dict(t=50, b=30, l=20, r=20))
# fig.write_image("images/kmeans_score_vs_k.pdf", scale=2, height=400, width=700)
fig

**Bacalah grafik di atas.** Di mana letak "siku"-nya? Perhatikan bahwa sumbu-x memakai skala logaritma. Setelah titik siku, penurunan inertia menjadi jauh lebih landai.

> **Catatan jujur.** Metode siku sering tidak memberikan jawaban yang tegas — kadang tidak ada siku yang jelas sama sekali. Karena itu pemilihan `k` dalam praktik juga mempertimbangkan pengetahuan domain dan kebutuhan aplikasinya.

---
*Kembali ke slide.*

---


---
## 5. Pendeteksi Wake Word

Bagian ini berpindah topik ke **probabilitas**.

*Wake word* adalah kata pembangun asisten suara, misalnya "Hei Siri" atau "OK Google". Perangkat terus mendengarkan dan berusaha mendeteksi kata itu.

Pertanyaannya: **jika detektor berbunyi, seberapa besar kemungkinan kata itu memang benar-benar diucapkan?** Ini pertanyaan yang berbeda dari "seberapa akurat detektornya", dan jawabannya sering mengejutkan.

Kita memakai **Teorema Bayes**:

\begin{align*}
P(W = 1 \mid D = 1) &= \frac{P(D=1 \mid W=1)\,P(W=1)}{P(D=1)}\\[6pt]
&= \frac{P(D=1 \mid W=1)\,P(W=1)}{P(D=1 \mid W=1)\,P(W=1) + P(D=1 \mid W=0)\,P(W=0)}
\end{align*}

**Arti setiap simbol:**

| Simbol | Arti | Istilah |
|---|---|---|
| $W = 1$ | Kata bangun benar-benar diucapkan | kejadian sebenarnya |
| $D = 1$ | Detektor berbunyi | hasil pengamatan |
| $P(W=1)$ | Peluang kata itu diucapkan pada suatu saat | **prior** |
| $P(D=1 \mid W=1)$ | Peluang detektor berbunyi saat kata memang diucapkan | **recall** / sensitivitas |
| $P(D=1 \mid W=0)$ | Peluang detektor berbunyi padahal kata tidak diucapkan | **false positive rate** |
| $P(W=1 \mid D=1)$ | Peluang kata memang diucapkan, bila detektor berbunyi | yang ingin dicari |

Penyebut $P(D=1)$ dihitung dengan **hukum probabilitas total**: detektor bisa berbunyi karena dua sebab — kata memang diucapkan, atau kata tidak diucapkan tetapi detektor keliru.

Mari kita terjemahkan perhitungan ini menjadi kode.


In [ ]:
def wake_word_detector(
        p_wake = 0.0001,            # P(W=1)      prior: peluang kata bangun diucapkan
        p_detect_g_wake = 0.99,     # P(D=1|W=1)  recall: deteksi benar
        p_detect_g_nowake = 0.001   # P(D=1|W=0)  false positive rate: deteksi palsu
):
    # Hukum probabilitas total untuk penyebutnya:
    # P(D=1) = P(D=1|W=1)P(W=1) + P(D=1|W=0)P(W=0)
    p_detect = p_wake * p_detect_g_wake + (1 - p_wake) * p_detect_g_nowake
    p_wake_g_detect = p_detect_g_wake * p_wake / p_detect
    return p_wake_g_detect

wake_word_detector()

### 5.1 Pengaruh Recall dan False Positive Rate

Sekarang kita amati apa yang terjadi bila mutu detektor kita ubah-ubah.

* **Recall** (disebut juga sensitivitas) adalah $P(D=1 \mid W=1)$, yaitu peluang detektor berhasil menangkap kata bangun ketika kata itu memang diucapkan.
* **False positive rate** adalah $P(D=1 \mid W=0)$, yaitu peluang detektor berbunyi padahal tidak ada kata bangun yang diucapkan.

Rumus Bayes tadi dapat kita tulis ulang sebagai:

$$
P(W = 1 \mid D=1) = \frac{(\text{Recall}) \cdot P(W=1)}{(\text{Recall}) \cdot P(W=1) + (\text{False Positive Rate}) \cdot P(W=0)}
$$

Dari persamaan ini terlihat sesuatu yang penting: **sekalipun recall mendekati 1 (detektor nyaris tidak pernah melewatkan kata bangun), peluang bahwa deteksi itu benar tetap bisa sangat rendah bila false positive rate-nya tinggi.**

Penyebabnya adalah nilai *prior* yang sangat kecil. Kata bangun jarang sekali diucapkan (bawaan pada fungsi kita: 0,0001), sementara detektor mendengarkan sepanjang waktu. Dengan demikian, kesempatan untuk melakukan kesalahan jauh lebih banyak daripada kesempatan untuk benar.

Dua grafik berikut memperlihatkan hal itu.


In [ ]:
p_detect_g_nowake = np.logspace(-6, -4, 100)
p_wake_g_detect = wake_word_detector(p_detect_g_nowake=p_detect_g_nowake,
                                     p_detect_g_wake=1.0)
fig = px.line(
    x=p_detect_g_nowake,
    y=p_wake_g_detect,
    title="P(W=1 | D=1) terhadap False Positive Rate, dengan Recall Sempurna",
    labels={
        "x": "P(D=1 | W=0) — False Positive Rate",
        "y": "P(W=1 | D=1)"
    },
    log_x=True
)
fig.update_layout( xaxis_exponentformat='power')
#fig.write_image("images/wake_word_detector_fpr.pdf", scale=2, height=500, width=700)
fig

**Bacalah grafik di atas.** Recall sudah disetel sempurna (bernilai 1,0), artinya detektor tidak pernah melewatkan satu pun kata bangun. Meskipun demikian, ketika false positive rate naik, peluang bahwa sebuah deteksi itu benar justru **anjlok tajam**.

Sekarang kita balik percobaannya: false positive rate ditahan tetap, lalu recall-nya yang divariasikan.

In [ ]:
p_detect_g_wake = np.logspace(-0.8, 0, 100)
p_wake_g_detect = wake_word_detector(p_detect_g_wake=p_detect_g_wake,
                                     p_detect_g_nowake=0.0001)
fig = px.line(
    x=p_detect_g_wake,
    y=p_wake_g_detect,
    title="P(W=1 | D=1) terhadap Recall, dengan False Positive Rate = 0,0001",
    labels={
        "x": "P(D=1 | W=1) — Recall",
        "y": "P(W=1 | D=1)"
    },
    log_x=True
)
fig.update_layout( xaxis_exponentformat='power')
#fig.write_image("images/wake_word_detector_recall.pdf", scale=2, height=500, width=700)
fig

**Bandingkan kedua grafik.** Perhatikan bahwa menaikkan recall hanya memberi perbaikan yang landai, sedangkan menurunkan false positive rate memberi perbaikan yang jauh lebih tajam.

**Inilah pelajaran utamanya:** pada persoalan deteksi kejadian yang langka, memperkecil peluang alarm palsu biasanya jauh lebih menentukan daripada memperbesar sensitivitas. Prinsip yang sama berlaku pada tes penyakit langka, deteksi penipuan kartu kredit, dan sistem deteksi penyusupan jaringan.

---
## 6. Penutup

### Ringkasan

**K-Means (unsupervised learning)**

* K-means mengelompokkan data **tanpa label** dengan mencari `k` titik pusat yang meminimalkan jumlah kuadrat jarak.
* Algoritma Lloyd bekerja dalam tiga tahap: inisialisasi, penugasan, dan pembaruan — dua tahap terakhir diulang sampai konvergen.
* Hasil k-means **bergantung pada inisialisasi**; menjalankannya beberapa kali dengan pusat awal berbeda adalah praktik yang lazim.
* K-means dapat dipakai jauh melampaui data dua dimensi: pada data piksel RGB, ia menjadi metode kompresi warna.
* Nilai `k` dipilih dengan metode siku, dengan catatan bahwa inertia selalu menurun seiring bertambahnya `k`.

**Probabilitas dan Teorema Bayes**

* Teorema Bayes membalik arah probabilitas bersyarat: dari $P(D \mid W)$ menjadi $P(W \mid D)$.
* Nilai **prior** yang sangat kecil dapat membuat sebuah deteksi tetap tidak meyakinkan, sekalipun detektornya sangat sensitif.
* Menurunkan **false positive rate** sering jauh lebih berdampak daripada menaikkan recall — pelajaran yang berlaku umum pada sistem deteksi kejadian langka: penyakit langka, penipuan kartu kredit, dan deteksi penyusupan.

### Latihan Mandiri

1. Jalankan `k_means_clustering(x, k)` beberapa kali **tanpa** menyetel `np.random.seed`. Apakah hasil pengelompokannya selalu sama? Apa artinya bagi keandalan k-means?
2. Ulangi pengelompokan data sepeda dengan `k = 2`, `3`, `5`, dan `6`. Buat plot inertia terhadap `k` dan tentukan letak sikunya. Apakah `k = 4` memang pilihan terbaik?
3. Kolom `Speed` dan `Length` memiliki rentang nilai yang berbeda. Terapkan `StandardScaler` sebelum k-means, lalu bandingkan hasilnya. Mengapa penskalaan penting bagi algoritma yang berbasis jarak?
4. Ubah `n_clusters` pada kompresi gambar menjadi 2, 4, 16, dan 32. Pada nilai berapa hasilnya mulai sulit dibedakan dari gambar asli menurut mata Anda?
5. Pada fungsi `wake_word_detector`, tetapkan recall = 0,99 dan false positive rate = 0,001. Berapa besar $P(W=1 \mid D=1)$? Lalu naikkan prior dari 0,0001 menjadi 0,01. Berapa kali lipat perubahannya, dan mengapa?
6. Sebuah tes penyakit memiliki recall 99% dan false positive rate 1%. Penyakit itu menjangkiti 1 dari 10.000 orang. Bila hasil tes seseorang positif, berapa peluang ia benar-benar sakit? Gunakan fungsi `wake_word_detector` untuk menghitungnya, lalu jelaskan mengapa hasilnya mengejutkan.

### Bacaan Lanjutan

* Dokumentasi k-means scikit-learn: https://scikit-learn.org/stable/modules/clustering.html#k-means
* Perbandingan algoritma klastering: https://scikit-learn.org/stable/modules/clustering.html
